In [2]:
import os
import sys
from dotenv import load_dotenv
from langchain_community.docstore.document import Document
from langchain_openai import ChatOpenAI
from langchain_core.retrievers import BaseRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import retrieval_qa
from sentence_transformers import CrossEncoder
from pydantic import Field,BaseModel
from utils.evaluate_rag import *
from utils.helper_functions import *

load_dotenv("/Users/nilasark/advanced/.env")
path="Understanding_Climate_Change.pdf"

In [5]:
vectorestore=encode_pdf(path)

In [10]:
class RatingScore(BaseModel):
    relevance_score:float=Field(...,description="The relevance score of a document to a query")
def rerank_documents(query:str,docs:list[Document],top_k:int=4):
    prompt_template=PromptTemplate(
    input_variables=["query", "doc"],
    template="""On a scale of 1-10, rate the relevance of the following document to the query. Consider the specific context and intent of the query, not just keyword matches.
        Query: {query}
        Document: {doc}
        Relevance Score:"""
        )

    llm=ChatOpenAI(model='gpt-4o-mini',temperature=0,max_completion_tokens=4000)
    llm_chain=prompt_template|llm.with_structured_output(schema=RatingScore)
    scored_docs=[]
    for doc in docs:
        content={"query":query,"doc":doc.page_content}
        ranked_result=llm_chain.invoke(content).relevance_score
        try:
            score=float(ranked_result)
        except ValueError:
            score=0
        scored_docs.append((doc,score))

    reranked_docs=sorted(scored_docs,key=lambda x:x[1],reverse=True)
    return [doc for doc,_ in reranked_docs[:top_k]]


In [ ]:
query="What are the impacts of climate change on biodiversity?"
initial_docs=vectorestore.similarity_search(query,k=3)
reranked_docs=rerank_documents(query,initial_docs,4)

print(f"Top Initial Documents:")
for i,doc in enumerate(initial_docs[:3]):
    print(f"Doc_{i}:{doc.page_content[:200]}...")

print(f"\nQuery:{query}")
print("\n")
for i,doc in enumerate(reranked_docs[:3]):
    print(f"Doc_{i}:{doc.page_content[:200]}...")